In [81]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from autobound.causalProblem import causalProblem
from autobound.DAG import DAG
from autobound.Query import Query

import io

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


$\mathcal{G}:$
```
          [R₃]         [R₁]     [R₂]
        ↙    ↘       ↙    ↘  ↙    ↘
       A ───→ B ───→ X ────→ M ────→ Y 
              ↑       │      ↑
              │       ↓      │
             [R₄]────→C─────→D
```

We have $\theta=(Y=1|do(X=1,M=1))$.

The mission of this notebook:
Calculate 
1. $a_1$: $pot(Y,M|do(X=1))$
2. $a_2$: $pot(Y,M|do(X=0))$


## Step 0: Simulate SCM

Remark: W.l.o.g. We can assume a simpler graph
$X \to M \to Y$ (@TODO Change that! Assume the actual graph)

In [45]:
N_samples = 1000000
np.random.seed(42)

# We are simulating a ground truth SCM. That is, we will 
# 1. randomly pick all root variables, 
# 2. and sample others as some arbitrary boolean function of their parents.
# We will simulate the actual confounders (not response types!) and we will make them binary. (That keeps the structural equations boolean)

# 1. Simulate Root variables (confounders, A)
A  = np.random.binomial(1, 0.3, N_samples)
U3 = np.random.binomial(1, 0.7, N_samples)
U4 = np.random.binomial(1, 0.6, N_samples)
U1 = np.random.binomial(1, 0.1, N_samples)
U2 = np.random.binomial(1, 0.8, N_samples)

# 2. Structural equations (B, X, C, D, M, Y)
B = A ^ U3
X = B ^ U1
C = X ^ U4
D = C ^ 1  # D = NOT(C)
M = (U1 & U2) ^ (X | D)
Y = U2 & M

ground_truth_data = pd.DataFrame({
    'A': A,
    'B': B,
    'X': X,
    'C': C,
    'D': D,
    'M': M,
    'Y': Y,
    'U1': U1,
    'U2': U2,
    'U3': U3,
    'U4': U4
})

## Step 1: Compute $\mathcal{W}_\emptyset$

### If we use the full DAG, the problem becomes computationally infeasible.

In [47]:
# Observational data
obs = ground_truth_data.drop(columns=['U1', 'U2', 'U3', 'U4'])
# compute probabilities for every combination of values (000, ... 111) and safe them as csv: A,B,X...,prob
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(6)
diff = round(1.0 - obs_counts['prob'].sum(), 6)

if diff != 0:
    idx = obs_counts['count'].idxmax()  # adjust the largest-mass row
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-4-obs.csv', index=False)

In [49]:
dag = DAG()
dag.from_structure("A->B, B->X, X->C, C->D, D->M, X->M, M->Y, " \
"U3->A, U3->B, U4->B, U4->C, U1->X, U1->M, U2->M, U2->Y", unob='U1,U2,U3,U4')
problem = causalProblem(dag)
problem.load_data('data/figure-4-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1, M=1)=1'))
program = problem.write_program()
program.to_pip('figure4.pip')

(lb, ub) = program.run_pyomo('ipopt', verbose=True, parallel=True)   # Converges to a locally infeasible point.
#(lb, ub) = program.run_pyomo('couenne', verbose=True, parallel=True)   # Runs forever
# (lb, ub) = program.run_couenne( verbose=True, epsilon=0.9, theta=0.9)   # Runs forever.
calW_empty = ub-lb

Ipopt 3.12.12: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.12, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:     1561
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:     4492

Total number of variables............................:      293
                     variables with only lower bounds:        0
                variables with lower and upper bounds:      293
                     variables with only upper bounds:        0
Tot

### That is why we set up the reduced program!
Recall that $R_\theta = \{R_2\}$ and therefore
* $D^*(R_\theta)= \{R_1, R_2, X, M, Y\}$.
* $D_\theta^* = \{\{R_1, R_2, X, M, Y\}\}$
* $R_\theta^* = \{R_1, R_2\}$

As shown in the paper, it suffices to optimize over $R_*^\theta$. Meaning we only have to model $D^*(R_\theta)$.

In [63]:
reduced_data = ground_truth_data.drop(columns=['U3', 'A', 'B', 'U4', 'C', 'D'])
obs = reduced_data.drop(columns=['U1', 'U2'])
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(40)

#make sure it's a valid distribution
if diff != 0:
    print(f"Adjusting for rounding error of {diff} by modifying the largest-mass row.")
    idx = obs_counts['count'].idxmax()  # adjust the largest-mass row
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-4-obs.csv', index=False)

In [139]:
dag = DAG()
dag.from_structure("X->M, M->Y, U1->X, U1->M, U2->M, U2->Y", unob='U1,U2')
problem = causalProblem(dag)
problem.load_data('data/figure-4-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1, M=1)=1'))
program = problem.write_program()
program.to_pip('figure4.pip')

(lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True) 
print(f"P(θ)∈[{lb:.2f}, {ub:.2f}]")
calW_empty = ub - lb
print(f"calW_empty = {calW_empty:.8f}")

P(θ)∈[0.57, 0.85]
calW_empty = 0.28538708


## Step 2: Compute $\mathcal{W_{a_1}}$
$a_1 = (Y,M|do(X=1))$

Summarizing my findings.
Intevening on M:
* $pot(Y|do(M=1)) > 0 (ID)$
* $pot(Y=0|do(M=1)) > 0$
* $pot(Y|do(M=0)) = 0$ (non-trivial! It can be >0 for some experiment outcomes $p$)

Intevening on X:
* $pot(Y,M |do(X=0))=0$
* $pot(Y,M |do(X=1))=0$

Intervening on both at the same time is equivalent to intervening on M.

In [114]:
# Add a single experiment P(Y=1, M=1 | do(X=1)) first.

def getIntervention(doX_val):
    M_doX = M = (U1 & U2) ^ (doX_val | D) #Set X=1 in structural equation
    Y_doX = U2 & M_doX #Note: If M_doX=0, Y_doX=0 always. So P(X=0,Y=1)=0
    doX = pd.DataFrame({
        'M_doX': M_doX,
        'Y_doX': Y_doX,
    })

    doX_counts = doX.value_counts().reset_index(name='count')

    # Round probabilities and force exact normalization after rounding
    doX_counts['prob'] = (doX_counts['count'] / N_samples).round(40)

    #make sure it's a valid distribution
    if diff != 0:
        print(f"Adjusting for rounding error of {diff} by modifying the largest-mass row.")
        idx = doX_counts['count'].idxmax()  # adjust the largest-mass row
        doX_counts.loc[idx, 'prob'] = round(doX_counts.loc[idx, 'prob'] + diff, 6)

    return doX_counts.drop(columns=['count'])
print(getIntervention(1))

   M_doX  Y_doX      prob
0      1      1  0.719872
1      1      0  0.199767
2      0      0  0.080361


In [ ]:
# Above we see the true experiment outcome. We do not know that! 
# That is why we will loop over all possible parameter values for p_00, p_01, p_10, p_11 to approximate W_a1(p_a1)
# For simplicty we will just loop over the interval [0, 1] for each of them! If the program is infeasible -> continue.
# we will probe values {0, 0.25, 0.5, 0.75, 1}. Note that the p vector needs to sum up to 1!

In [ ]:
def W_X(p_config, X_val):
    # Calculate witdh under X=X_val for given p_configuration
    problem = causalProblem(dag)
    problem.load_data('data/figure-4-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1, M=1)=1'))

    for _, row in p_config.iterrows():
        m, y = int(row.M_doX), int(row.Y_doX)
        lhs = problem.query(f'M(X={X_val})={m}&Y(X={X_val})={y}')
        problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb




def calW_X(X_val, granularity):
    # Loop over all possible probability vectors (p_00, p_01, p_10, p_11) that sum to 1.
    # Granularity n means we probe multiples of 1/n, i.e. {0, 1/n, 2/n, ..., 1}.
    n = granularity  # granularity: n=4 yields steps of 0.25

    W_approx = []

    for a in range(n + 1):
        for b in range(n + 1 - a):
            for c in range(n + 1 - a - b):
                d = n - a - b - c
                p_00, p_01, p_10, p_11 = a / n, b / n, c / n, d / n

                doX_hyp = pd.DataFrame({
                    'M_doX': [1, 1, 0, 0],
                    'Y_doX': [1, 0, 0, 1],
                    'prob':  [p_11, p_10, p_00, p_01]
                })
                try: # use try catch to skip all infeasible configurations
                    W_approx.append(W_X(doX_hyp, X_val))
                except Exception as e:
                    # Skip infeasible/failed solver runs
                    print(f"Skipping p=({p_00:.2f}, {p_01:.2f}, {p_10:.2f}, {p_11:.2f}): {e}")
                    continue
    return max(W_approx)
            

In [129]:
calW_1 = calW_X(1, 8)
calW_0 = calW_X(0, 8)

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpl97pepow.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpadixookt.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpk42oaqq3.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmp4pmhzwa5.pyomo.nl Mar 12 2026)\x3a
      Infeasible
ERROR: evaluating object as numeric value: objvar
        (object: <class 'pyomo.core.base.var.ScalarVar'>)
    No value for uninitialized ScalarVar object objvar
Skipping p=(0.00, 0.00, 0.00, 1.00): No value for uninitialized ScalarVar object objvar
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpocyk5ooq.py

In [141]:
print(f"W_empty = {calW_empty:.8f}")
print(f"W_0 = {calW_0:.8f}, W_1 = {calW_1:.8f}")
print(f"pot(Y,M|do(X=1))={calW_empty - calW_1:.8f}, pot(Y,M|do(X=0))={calW_empty - calW_0:.8f}")

W_empty = 0.28538708
W_0 = 0.28538708, W_1 = 0.28538707
pot(Y,M|do(X=1))=0.00000001, pot(Y,M|do(X=0))=-0.00000000


In [165]:
def W_single(p_y, X_val=None, M_val=None, Y_val=1):
    problem = causalProblem(dag)
    problem.load_data('data/figure-4-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1, M=1)=1'))

    if X_val is not None:
        interv = f'X={X_val}'
    else:
        interv = f'M={M_val}'

    lhs = problem.query(f'Y({interv})={Y_val}')
    problem.add_constraint(lhs - Query(p_y))

    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb


def calW_single(granularity, X_val=None, M_val=None, Y_val=1):
    n = granularity
    W_approx = []

    for a in range(n + 1):
        p_y = a / n
        try:
            W_approx.append(W_single(p_y, X_val=X_val, M_val=M_val, Y_val=Y_val))
        except Exception as e:
            print(f"Skipping p_y={p_y:.2f}: {e}")
            continue
    return max(W_approx)


In [171]:
calW_single(8, Y_val=1, X_val=1)

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpc0jkppa5.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmp_kgsy8xm.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmp73kg93h2.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpv2qv887r.pyomo.nl Mar 12 2026)\x3a
      Infeasible
ERROR: evaluating object as numeric value: objvar
        (object: <class 'pyomo.core.base.var.ScalarVar'>)
    No value for uninitialized ScalarVar object objvar
Skipping p_y=0.00: No value for uninitialized ScalarVar object objvar
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpuytwgypa.pyomo.nl Mar 12 2026

0.28538700000000006

In [146]:
def W_M(p_config, M_val):
    # Calculate width under M=M_val for given p_configuration
    problem = causalProblem(dag)
    problem.load_data('data/figure-4-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1, M=1)=1'))

    for _, row in p_config.iterrows():
        y = int(row.Y_doM)
        lhs = problem.query(f'Y(M={M_val})={y}')
        problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb


def calW_M(M_val, granularity):
    # Loop over all possible probability vectors (p_0, p_1) that sum to 1.
    n = granularity

    W_approx = []

    for a in range(n + 1):
        b = n - a
        p_0, p_1 = a / n, b / n

        doM_hyp = pd.DataFrame({
            'Y_doM': [0, 1],
            'prob':  [p_0, p_1]
        })
        try:
            W_approx.append(W_M(doM_hyp, M_val))
        except Exception as e:
            print(f"Skipping p=({p_0:.2f}, {p_1:.2f}): {e}")
            continue
    print(W_approx)
    return max(W_approx)

In [150]:
calW_M(0, 4)

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmplf3gh5ld.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpphpdzoil.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpt2qgpe37.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmp46lgsa_a.pyomo.nl Mar 12 2026)\x3a
      Infeasible
ERROR: evaluating object as numeric value: objvar
        (object: <class 'pyomo.core.base.var.ScalarVar'>)
    No value for uninitialized ScalarVar object objvar
Skipping p=(0.00, 1.00): No value for uninitialized ScalarVar object objvar
[0.0432462553664722, 0.2853870786104675, 0.2853870787371714, 0.28538703996000014]


0.2853870787371714

In [151]:
def W_XM(p_config, X_val, M_val):
    problem = causalProblem(dag)
    problem.load_data('data/figure-4-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1, M=1)=1'))

    for _, row in p_config.iterrows():
        y = int(row.Y_doXM)
        lhs = problem.query(f'Y(X={X_val}, M={M_val})={y}')
        problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb


def calW_XM(X_val, M_val, granularity):
    n = granularity

    W_approx = []

    for a in range(n + 1):
        b = n - a
        p_0, p_1 = a / n, b / n

        doXM_hyp = pd.DataFrame({
            'Y_doXM': [0, 1],
            'prob':   [p_0, p_1]
        })
        try:
            W_approx.append(W_XM(doXM_hyp, X_val, M_val))
        except Exception as e:
            print(f"Skipping p=({p_0:.2f}, {p_1:.2f}): {e}")
            continue
    return max(W_approx)


In [155]:
calW_XM(0,1,4)

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpu4guxk6p.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpv9y49c3h.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmp7ei82ghi.pyomo.nl Mar 12 2026)\x3a
      Infeasible
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpjaf4pg2o.pyomo.nl Mar 12 2026)\x3a
      Infeasible
ERROR: evaluating object as numeric value: objvar
        (object: <class 'pyomo.core.base.var.ScalarVar'>)
    No value for uninitialized ScalarVar object objvar
Skipping p=(0.00, 1.00): No value for uninitialized ScalarVar object objvar
model.name="unknown";
    - termination condition: infeasible
    - message from solver: Couenne (/tmp/tmpwhdyy9a7.pyomo.nl Mar 1

0.0